# Riemannian score-based generative models on SO(3)^J: a tutorial

This notebook explains, step by step and with runnable code, the **Riemannian score-based generative model (RSGM)** of this repository (`src/hp/models/rsgm.py`, `src/hp/models/igso3.py`), a diffusion model for human poses.

**The problem.** A body pose of the SMPL model is a rotation per joint: 19 rotations here (the joints of `AmassJoints4Learning`), so a pose is a point of the product space $SO(3)^{19}$, a curved space (a manifold) of dimension $3 \cdot 19 = 57$. We want a **pose prior**: a probability distribution $p(x)$ over poses learned from motion capture data (AMASS), which we can
1. **sample** from (generate new, realistic poses),
2. **evaluate**: give the (log-)density $\log p(x)$ of a pose (how plausible is it?),
3. **differentiate**: $\nabla_x \log p(x)$, to push a pose towards plausible ones (e.g. when fitting a body to an image).

**The idea of a score-based model** (Song et al. 2021; on manifolds: De Bortoli et al. 2022, "Riemannian Score-Based Generative Modelling"):
- *Noising:* take a data pose and let it diffuse by Brownian motion until it is completely random (uniform on $SO(3)^J$).
- *Learning:* train a neural network to predict, at every noise level, the **score** $\nabla \log p_t(x)$: the direction towards more probable poses.
- *Generating:* start from a uniform random pose and run the noising *backwards in time*, steered by the learned score.
- *Likelihood:* an ordinary differential equation (the probability flow) with the same marginals turns the score into exact log-densities.

**Outline**
1. Rotations: the geometry of SO(3)
2. Brownian motion on SO(3) and its heat kernel IGSO(3)
3. The noising process of RSGM
4. The score and the denoising score matching loss
5. The network
6. Training on a toy distribution
7. Sampling: the time reversal
8. The likelihood: probability flow ODE
9. The real thing: a pose prior trained on AMASS
10. Exercises

Everything except section 9 runs on a CPU in a few minutes. Section 9 needs a trained run (from `train_rsgm.ipynb`) and is skipped otherwise.

In [ ]:
import inspect
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import roma
import torch

from hp.models import build_model, igso3
from hp.models.rsgm import RSGM, exp_frame
from hp.training import TrainConfig, Trainer

torch.manual_seed(0)
np.random.seed(0)
DEVICE = "cpu"   # the toy model is small; "mps" or "cuda:0" also work


def show_source(obj):
    # Print the source code of a function or class of the package.
    print(inspect.getsource(obj))


def rotvec_scatter(ax, rotations, title, color="C0", s=2, alpha=0.3):
    # Rotations [N, 3, 3] as rotation vectors (axis * angle) in the ball of radius pi.
    v = roma.rotmat_to_rotvec(torch.as_tensor(rotations, dtype=torch.float64)).numpy()
    ax.scatter(v[:, 0], v[:, 1], v[:, 2], s=s, alpha=alpha, color=color)
    for set_lim in (ax.set_xlim, ax.set_ylim, ax.set_zlim):
        set_lim(-math.pi, math.pi)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]), ax.set_yticks([]), ax.set_zticks([])

## 1. Rotations: the geometry of SO(3)

A rotation is a $3 \times 3$ matrix $R$ with $R^\top R = I$ and $\det R = 1$. The set of them, $SO(3)$, is a 3-dimensional manifold and a group (rotations compose by matrix multiplication).

**Axis-angle and the exponential map.** Every rotation turns by an angle $\omega \in [0, \pi]$ about a unit axis $u$. The **rotation vector** $v = \omega u \in \mathbb{R}^3$ describes it, and the matrix is $R = \exp(\hat v)$, where $\hat v$ is the skew-symmetric matrix with $\hat v w = v \times w$ (the matrix exponential is Rodrigues' formula; `roma.rotvec_to_rotmat`). The inverse is the logarithm, $v = \log(R)^\vee$ (`roma.rotmat_to_rotvec`). The skew matrices $\hat v$ form the **Lie algebra** $\mathfrak{so}(3) \cong \mathbb{R}^3$.

**Distance.** We use the metric for which the geodesic distance of two rotations is the angle of the rotation between them:
$$d(R_1, R_2) = \angle(R_1^\top R_2) = \arccos\frac{\operatorname{tr}(R_1^\top R_2) - 1}{2}.$$

**Tangent vectors in the frame of the rotation.** A tangent vector at $x \in SO(3)$ is written $x\,\hat v$ with coordinates $v \in \mathbb{R}^3$, and its length is $|v|$. Moving from $x$ along it for unit time gives $x \exp(\hat v)$. This is everything the code needs, and it is the function `exp_frame`:

In [ ]:
show_source(exp_frame)

x = roma.random_rotmat(1, generator=torch.Generator().manual_seed(1))[0].double()
v = torch.tensor([0.0, 0.0, 0.3], dtype=torch.float64)   # 0.3 rad about the z axis of the frame of x
y = exp_frame(x, v)
angle = torch.arccos(((x.T @ y).trace() - 1) / 2)
print(f"distance of x and exp_frame(x, v): {angle:.4f} rad = |v| = {v.norm():.4f}")

**The uniform distribution on SO(3)** (the normalized Haar measure) is the "flat" distribution: it does not prefer any rotation. Its rotation angle is *not* uniform on $[0, \pi]$: there are more rotations with large angles than small ones (the "sphere" of rotations with angle $\omega$ has area $\propto 1 - \cos\omega$), so the angle has the density
$$p(\omega) = \frac{1 - \cos\omega}{\pi}, \qquad \omega \in [0, \pi].$$
All densities in this notebook are **relative to this uniform distribution**: a density of 1 (log-density 0) means "as likely as a uniformly random rotation".

In [ ]:
uniform = roma.random_rotmat(100_000, generator=torch.Generator().manual_seed(0)).double()
angles = roma.rotmat_to_rotvec(uniform).norm(dim=-1)
grid = np.linspace(0, math.pi, 200)

fig = plt.figure(figsize=(11, 3.8))
ax = fig.add_subplot(1, 2, 1)
ax.hist(angles.numpy(), bins=80, density=True, alpha=0.6, label="uniform random rotations")
ax.plot(grid, (1 - np.cos(grid)) / math.pi, "k", label=r"$(1-\cos\omega)/\pi$")
ax.set_xlabel(r"rotation angle $\omega$")
ax.legend()
rotvec_scatter(fig.add_subplot(1, 2, 2, projection="3d"), uniform[:5000], "uniform rotations as rotation vectors")
fig.tight_layout()
plt.show()

## 2. Brownian motion on SO(3) and its heat kernel IGSO(3)

**Brownian motion** is the continuous random walk. On SO(3) it is simplest as a *geodesic random walk*: many tiny steps in random directions,
$$x_{k+1} = x_k \exp\big(\widehat{\sqrt{\Delta s}\, z_k}\big), \qquad z_k \sim \mathcal{N}(0, I_3).$$
Its generator is half the Laplace-Beltrami operator, $\tfrac12\Delta$. After a time $s$, started at $x_0$, the rotation is $x_0 \exp(\widehat{\omega u})$ with a uniformly random axis $u$ and a random angle $\omega$. This distribution is the **isotropic Gaussian on SO(3)**, IGSO(3), and its density (relative to uniform) depends only on the angle $\omega$ between $x$ and $x_0$. It is the **heat kernel** of SO(3), known in closed form as a series over the irreducible representations:
$$f(\omega; s) = \sum_{l=0}^{\infty} (2l+1)\, e^{-l(l+1)s/2}\, \frac{\sin((l+\tfrac12)\omega)}{\sin(\omega/2)}.$$

For small $s$ the series converges very slowly (thousands of terms at $s = 10^{-4}$), and the diffusion model needs tiny $s$. Poisson summation turns it into a sum over the *images* of a Gaussian, which converges fast for every $s$ we need:
$$f(\omega; s) = e^{s/8} \sqrt{\tfrac{2\pi}{s}}\, \frac{1}{s} \sum_{k\in\mathbb{Z}} (-1)^k\, (\omega + 2\pi k)\, \frac{e^{-(\omega+2\pi k)^2/(2s)}}{\sin(\omega/2)}.$$
For small $s$ only $k = 0$ matters: then IGSO(3) is a Gaussian in the rotation vector, $\propto e^{-\omega^2/(2s)}$, with a geometric correction factor $\omega / \sin(\omega/2)$.

`igso3.log_density` implements the image sum (with care for numerical underflow), `igso3.log_density_series` the series (as a reference):

In [ ]:
show_source(igso3._images)
show_source(igso3.log_density)

In [ ]:
omega = torch.linspace(0.01, math.pi - 0.01, 400, dtype=torch.float64)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for s, color in zip([0.02, 0.1, 0.5, 2.0], ["C0", "C1", "C2", "C3"]):
    density = igso3.log_density(omega, s).exp() * (1 - torch.cos(omega)) / math.pi  # the density of the ANGLE
    axes[0].plot(omega, density, color=color, label=f"s = {s}")
    if s >= 0.5:  # where the series converges with 2000 terms
        diff = (igso3.log_density(omega, s) - igso3.log_density_series(omega, s)).abs().max()
        print(f"s = {s}: image sum vs series, max |difference of log f| = {diff:.1e}")
axes[0].set_xlabel(r"angle $\omega$ to $x_0$")
axes[0].set_ylabel("density of the angle")
axes[0].legend()
axes[0].set_title("IGSO(3): the angle after Brownian motion for a time s")

# a geodesic random walk ends in IGSO(3) with the same s: the time convention of the code
s, steps, n = 0.5, 200, 20_000
walk = torch.eye(3, dtype=torch.float64).expand(n, 3, 3)
gen = torch.Generator().manual_seed(0)
for _ in range(steps):
    walk = walk @ roma.rotvec_to_rotmat(math.sqrt(s / steps) * torch.randn(n, 3, dtype=torch.float64, generator=gen))
axes[1].hist(roma.rotmat_to_rotvec(walk).norm(dim=-1).numpy(), bins=80, density=True, alpha=0.6, label=f"random walk, {steps} steps")
axes[1].plot(omega, igso3.log_density(omega, s).exp() * (1 - torch.cos(omega)) / math.pi, "k", label="IGSO(3) density")
axes[1].set_xlabel(r"angle $\omega$")
axes[1].set_title(f"Brownian motion for s = {s}")
axes[1].legend()
fig.tight_layout()
plt.show()

**Sampling IGSO(3).** To noise data we must draw from IGSO(3) for any $s$, fast. The axis is uniform on the sphere (`igso3.random_axes`: a normalized Gaussian vector). The angle is drawn by inverting its cumulative distribution function: `igso3.AngleSampler` tabulates the quantile function once on a grid of $\log s$ and of probabilities, and interpolates. It is checked by a Kolmogorov-Smirnov test in `tests/test_igso3.py`:

In [ ]:
sampler = igso3.AngleSampler(1e-6, 6.0)
fig, ax = plt.subplots(figsize=(6, 3.5))
for s, color in zip([1e-4, 0.01, 0.3, 3.0], ["C0", "C1", "C2", "C3"]):
    drawn = sampler.sample(torch.full((20_000,), s), torch.Generator().manual_seed(0))
    ks = (igso3.angle_cdf(drawn.sort().values[::40], s) - torch.arange(1, 20_001, 40, dtype=torch.float64) / 20_000).abs().max()
    ax.hist(np.log10(drawn.numpy()), bins=80, density=True, alpha=0.5, color=color, label=f"s = {s:g} (KS distance {ks:.3f})")
ax.set_xlabel(r"$\log_{10}$ angle")
ax.set_title("angles drawn by AngleSampler")
ax.legend(fontsize=8)
plt.show()

## 3. The noising process of RSGM

Every joint of the pose diffuses independently by Brownian motion, with a time-dependent speed $\beta(t)$ for $t \in [0, 1]$:
$$dx_t = \sqrt{\beta(t)}\, dB_t, \qquad \beta(t) = \beta_0 + (\beta_f - \beta_0)\, t.$$
This is Brownian motion run with the clock $s(t) = \int_0^t \beta = \beta_0 t + \tfrac12(\beta_f - \beta_0) t^2$, so the noised pose is known in closed form:
$$x_t \mid x_0 \;\sim\; \text{IGSO}(3)\big(x_0,\, s(t)\big) \text{ in every joint.}$$
At $t = 1$, $s(1) \approx \beta_f / 2$: with $\beta_f = 10$ the pose is uniform up to $e^{-5}$ in the slowest mode, so the model starts generation from the uniform distribution. (The JAX code of the paper has $\beta_f = 6$; then the end is visibly not uniform, which biases the likelihood by ~0.01 nats per joint.)

The model keeps the schedule in three small methods:

In [ ]:
for method in (RSGM.beta, RSGM.s, RSGM.std):
    show_source(method)

**A toy distribution.** To see everything at work we use a single joint ($J = 1$) and a distribution we know exactly: a mixture of three IGSO(3) "blobs" around three rotations $\mu_k$,
$$p_0(x) = \sum_{k=1}^{3} w_k\, f\big(\angle(\mu_k^\top x);\, \sigma_k\big).$$
Because Brownian motion just adds time, the noised distribution is again such a mixture, with the times $\sigma_k + s(t)$. So we know the true $p_t$ and its score at every $t$: a perfect check for what the model learns.

In [ ]:
class ToyMixture:
    # A mixture of IGSO(3) distributions on SO(3) (one joint): means mu [K, 3, 3], times sigma [K], weights w [K].

    def __init__(self, mu, sigma, w):
        self.mu, self.sigma, self.w = mu.double(), torch.as_tensor(sigma, dtype=torch.float64), torch.as_tensor(w, dtype=torch.float64)
        self.angles = igso3.AngleSampler(1e-4, 20.0)

    def sample(self, n, generator=None):
        k = torch.multinomial(self.w, n, replacement=True, generator=generator)
        omega = self.angles.sample(self.sigma[k], generator)
        return exp_frame(self.mu[k], omega[:, None] * igso3.random_axes((n,), generator))  # [n, 3, 3]

    def log_prob(self, x, s=0.0):
        # log p_s(x) relative to uniform, after noising for an extra Brownian time s (x: [n, 3, 3]).
        cos = ((self.mu[None].transpose(-1, -2) @ x[:, None]).diagonal(dim1=-2, dim2=-1).sum(-1) - 1) / 2
        omega = torch.arccos(cos.clamp(-1 + 1e-12, 1 - 1e-12))                   # [n, K] angles to the means
        return torch.logsumexp(self.w.log() + igso3.log_density(omega, self.sigma + s), dim=1)


mu = roma.random_rotmat(3, generator=torch.Generator().manual_seed(3))
toy = ToyMixture(mu, sigma=[0.02, 0.05, 0.1], w=[0.5, 0.3, 0.2])
data = toy.sample(20_000, torch.Generator().manual_seed(0))
print(f"toy data: {tuple(data.shape)}, mean log-density {toy.log_prob(data[:2000]).mean():.2f} (uniform: 0)")

How to read the 3D pictures: a rotation is drawn as its rotation vector $\omega u$, a point in the ball of radius $\pi$. On the surface of the ball, $v$ and $-v$ (angle $\pi$ about opposite axes) are **the same rotation**: the ball is glued to itself at antipodal points. So a blob whose mean has an angle close to $\pi$ appears cut into two halves on opposite sides of the ball: below, three blobs can look like four clusters.

In [ ]:
noiser = build_model("rsgm", num_joints=1, hidden=[8], beta_f=10.0)  # only its schedule is used here
times = [0.0, 0.05, 0.15, 0.3, 0.6, 1.0]
x0 = data[:3000]

fig = plt.figure(figsize=(15, 5.3))
for i, t in enumerate(times):
    s = float(noiser.s(torch.tensor(t)))
    if t == 0:
        xt = x0
    else:
        omega = noiser.angles.sample(torch.full((len(x0),), s))
        xt = exp_frame(x0, omega[:, None] * igso3.random_axes((len(x0),)))
    rotvec_scatter(fig.add_subplot(2, len(times), i + 1, projection="3d"), xt, f"t = {t}, s(t) = {s:.3f}")
t_grid = torch.linspace(1e-3, 1, 200)
ax = fig.add_subplot(2, 2, 3)
ax.plot(t_grid, noiser.beta(t_grid), label=r"$\beta(t)$")
ax.plot(t_grid, noiser.s(t_grid), label=r"$s(t) = \int_0^t \beta$")
ax.set_xlabel("t")
ax.legend()
ax = fig.add_subplot(2, 2, 4)
ax.semilogy(t_grid, noiser.std(t_grid), label=r"std$(t) = \sqrt{1 - e^{-s(t)}}$ (scale of the network output)")
ax.set_xlabel("t")
ax.legend()
fig.suptitle("Forward noising of the toy data: the three blobs spread until the rotations are uniform")
fig.tight_layout()
plt.show()

## 4. The score and the denoising score matching loss

The **score** of a density $p$ on a manifold is its Riemannian gradient $\nabla \log p(x)$, a tangent vector that points to more probable poses. In the frame of the rotation it is a vector $\xi \in \mathbb{R}^3$ per joint.

**The score of IGSO(3).** The density depends only on the angle $\omega$ to $x_0$. If $x = x_0 \exp(\widehat{\omega u})$, moving $x$ by $x \exp(\hat\delta)$ changes the angle by $u \cdot \delta$ to first order, so
$$\nabla_x \log f = \frac{d \log f}{d\omega}(\omega; s)\; u \quad\text{(in the frame of } x\text{)}.$$
It points along the axis, and its sign is negative: towards $x_0$. `igso3.d_log_density` is the derivative in closed form; for small $s$ it is $\approx -\omega/s$ (the Gaussian score).

**Denoising score matching.** We do not know the score of the *data* distribution $p_t$, but we know the score of the noising kernel $p_t(x_t \mid x_0)$. A classical identity (Vincent 2011) says that regressing a network onto the kernel's score gives the score of $p_t$:
$$\mathcal{L}(\theta) = \mathbb{E}_{t,\, x_0,\, x_t \mid x_0}\Big[\big\|\, \text{std}(t)\big(s_\theta(x_t, t) - \nabla \log p_t(x_t \mid x_0)\big)\big\|^2\Big].$$
The weight $\text{std}(t)^2 = 1 - e^{-s(t)}$ balances the noise levels (the score is $\sim 1/\sqrt{s}$ large near $t = 0$). One training step:
1. draw $t \sim U[t_\min, 1]$, and an angle $\omega \sim$ IGSO(3)$(s(t))$ and an axis $u$ per joint;
2. noise: $x_t = x_0 \exp(\widehat{\omega u})$;
3. target: $\frac{d\log f}{d\omega}(\omega; s(t))\, u$, known exactly;
4. regress the network on it.

Here is `RSGM.loss`. Note that the noising runs on the CPU in float64: near $t = 0$ the target is $\sim 1/s$ large and float32 is not precise enough. In eval mode the times and noise come from a fixed seed, so validation losses are comparable.

In [ ]:
show_source(RSGM.loss)

In [ ]:
# the target points towards x0 and equals the derivative of the log-density along the axis
s = torch.tensor(0.05, dtype=torch.float64)
omega = torch.linspace(0.01, 1.0, 200, dtype=torch.float64)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(omega, igso3.d_log_density(omega, s), label=r"$d \log f / d\omega$ (closed form)")
ax.plot(omega, -omega / s, "--", label=r"$-\omega / s$ (Gaussian, small s)")
h = 1e-6
ax.plot(omega[::10], ((igso3.log_density(omega + h, s) - igso3.log_density(omega - h, s)) / (2 * h))[::10], "k.", label="finite differences")
ax.set_xlabel(r"angle $\omega$ to $x_0$")
ax.set_title(f"score of IGSO(3) along the axis, s = {float(s)}")
ax.legend()
plt.show()

## 5. The network

`RSGM.score` is a plain MLP (5 layers of 512 with sine activations in the paper's setting) that gets the $9J$ entries of the rotation matrices and the time $t$, and outputs $3J$ numbers: the score's coordinates in the frames of the joints (the "Lie algebra" parametrization of the paper). The output is divided by std$(t)$, so the network itself only has to predict a quantity of order 1 at every noise level (the *std trick* of Song & Ermon).

Why rotation matrices as input? They are a continuous, unique representation of a rotation (axis-angle and quaternions have jumps or double covers), and an MLP handles them well.

In [ ]:
show_source(RSGM.score)
model_full = build_model("rsgm", num_joints=19)
print(model_full.net)
print(f"{model_full.num_parameters():,} parameters for J = 19")

## 6. Training on the toy distribution

The training loop is the generic `hp.training.Trainer` of the model zoo: Adam, gradient clipping and an exponential moving average (EMA) of the weights, which is what is used for sampling and likelihood. It only calls `model.loss(batch)`. For the toy we use a smaller network and a larger learning rate than for real poses.

In [ ]:
toy_model = build_model("rsgm", num_joints=1, hidden=[256, 256, 256], beta_f=10.0, sample_steps=200, likelihood_steps=200)
train_set = data[:18_000].reshape(-1, 1, 3, 3).float()
valid_set = data[18_000:].reshape(-1, 1, 3, 3).float()
config = TrainConfig(max_iterations=4000, batch_size=512, lr=1e-3, scheduler="none", ema_decay=0.999,
                     valid_interval=500, checkpoint_every=0, device=DEVICE, progress=False)
start = time.perf_counter()
trainer = Trainer(toy_model, train_set, valid_set, config)
history = trainer.fit()
toy_model = trainer.ema.model.eval()  # the EMA weights
print(f"trained {config.max_iterations:,} steps in {time.perf_counter() - start:.0f} s")
for step, value in zip(history.valid_iteration, history.valid_loss):
    print(f"  step {step:5d}: validation loss {value:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
loss = np.asarray(history.loss)
ax.plot(np.convolve(loss, np.ones(100) / 100, mode="valid"), label="training (moving average)")
ax.plot(history.valid_iteration, history.valid_loss, "o-", label="validation (EMA weights, fixed noise)")
ax.set_xlabel("step")
ax.set_ylabel("denoising score matching loss")
ax.legend()
plt.show()

Did the network learn the score? For the toy we know the true score of $p_t$ (of the noised mixture), so we can compare at a few noise levels. Below: the cosine between learned and true score, and their lengths, at noised data points.

In [ ]:
def log_map(r):
    # Rotation vectors of rotations [..., 3, 3] in plain ops, so that torch.func can differentiate through it.
    # Near an angle of pi the axis is ill-defined (the skew part vanishes); there the IGSO(3) score is ~0 anyway
    # (the density is flat at pi), so the clamps only keep the numbers finite.
    angle = torch.arccos(((r.diagonal(dim1=-2, dim2=-1).sum(-1) - 1) / 2).clamp(-1 + 1e-7, 1 - 1e-7))
    axis = torch.stack([r[..., 2, 1] - r[..., 1, 2], r[..., 0, 2] - r[..., 2, 0], r[..., 1, 0] - r[..., 0, 1]], -1)
    return axis / axis.norm(dim=-1, keepdim=True).clamp_min(1e-12) * angle[..., None]


def true_score(x, t, mixture=toy, schedule=toy_model):
    # The exact score of the noised toy mixture at x [n, 1, 3, 3], in the frame of x: a weighted sum of the
    # IGSO(3) scores of the components, weighted by their posterior probabilities.
    s = schedule.s(t).double()[:, None]                                     # [n, 1]
    v = log_map(mixture.mu[None].transpose(-1, -2) @ x.double())            # [n, K, 3]: x relative to each mean
    omega = v.norm(dim=-1)
    log_w = mixture.w.log() + igso3.log_density(omega, mixture.sigma + s)  # [n, K]
    posterior = torch.softmax(log_w, dim=1)
    scores = (igso3.d_log_density(omega, mixture.sigma + s) / omega)[..., None] * v
    return (posterior[..., None] * scores).sum(1)[:, None, :]              # [n, 1, 3]


print(f"{'t':>6s} {'cos(learned, true)':>20s} {'|learned| / |true|':>20s}")
x0 = data[18_000:19_000].reshape(-1, 1, 3, 3)
for t_value in [0.01, 0.05, 0.2, 0.5, 0.9]:
    t = torch.full((len(x0),), t_value)
    s = toy_model.s(t).double()
    omega = toy_model.angles.sample(s[:, None])
    xt = exp_frame(x0, omega[..., None] * igso3.random_axes((len(x0), 1)))
    with torch.no_grad():
        learned = toy_model.score(xt.float(), t).double()
    true = true_score(xt, t)
    cos = (learned * true).sum(-1) / (learned.norm(dim=-1) * true.norm(dim=-1))
    print(f"{t_value:6.2f} {cos.median():20.3f} {(learned.norm(dim=-1) / true.norm(dim=-1)).median():20.3f}")

The directions are right at every noise level. The lengths are hardest to learn at the smallest $t$: there the regression targets $\nabla \log p_t(x_t \mid x_0) \approx -\omega/s$ are huge and noisy (they point to the *particular* $x_0$ the sample came from), while their average, the score of $p_t$, is much smaller (it points to the blob as a whole). The network has to average out a lot of noise, which takes many steps. Exercise 3 explores how the weighting std$(t)^2$ of the loss affects this regime.

## 7. Sampling: the time reversal

A remarkable fact (Anderson 1982; on manifolds De Bortoli et al. 2022): the noising process run backwards in time is again a diffusion, whose drift is the score:
$$dx = \beta(t)\, \nabla \log p_t(x)\, dt + \sqrt{\beta(t)}\, d\bar B_t, \qquad t: 1 \to 0.$$
So: start from a uniform random pose at $t = 1$ and integrate backwards with the learned score. On the manifold the simplest integrator is the **geodesic Euler-Maruyama** step: compute the (deterministic + random) tangent step in the frame of $x$ and follow it with the exponential map,
$$x \leftarrow x \exp\Big(\widehat{\beta(t)\, s_\theta(x, t)\, \Delta t + \sqrt{\beta(t)\Delta t}\; z}\Big).$$
Each step keeps $x$ exactly on SO(3), unlike steps in $\mathbb{R}^9$ followed by a projection. The sampler stops at a small $t = \varepsilon$ ($2 \cdot 10^{-4}$), where the noise is far below a degree.

In [ ]:
show_source(RSGM.sample)

The same loop, but keeping the intermediate states, shows how uniform noise turns into the three blobs. Compare with the forward noising of section 3: it is the same film played backwards.

In [ ]:
@torch.no_grad()
def sample_with_history(model, n, keep_at=(1.0, 0.6, 0.3, 0.15, 0.05, 0.0)):
    x = roma.random_rotmat(n * model.num_joints).reshape(n, model.num_joints, 3, 3)
    times = torch.linspace(1.0, model.eps, model.sample_steps + 1)
    history = {1.0: x.clone()}
    for t, t_next in zip(times[:-1], times[1:]):
        dt, beta = t - t_next, model.beta(t)
        step = beta * model.score(x, t.expand(n)) * dt + torch.sqrt(beta * dt) * torch.randn(n, model.num_joints, 3)
        x = exp_frame(x, step)
        for keep in keep_at:
            if keep not in history and t_next <= max(keep, model.eps):
                history[keep] = x.clone()
    return x, history


torch.manual_seed(0)
generated, history = sample_with_history(toy_model, 3000)
fig = plt.figure(figsize=(15, 2.8))
for i, t in enumerate(sorted(history, reverse=True)):
    rotvec_scatter(fig.add_subplot(1, len(history), i + 1, projection="3d"), history[t][:, 0], f"t = {t}", color="C1")
fig.suptitle("Generation: the time reversal from uniform rotations (t = 1) to the data distribution (t = 0)")
fig.tight_layout()
plt.show()

fig = plt.figure(figsize=(8, 4))
rotvec_scatter(fig.add_subplot(1, 2, 1, projection="3d"), data[:3000], "toy data")
rotvec_scatter(fig.add_subplot(1, 2, 2, projection="3d"), generated[:, 0], "generated", color="C1")
fig.tight_layout()
plt.show()

A quantitative check: which blob does a sample belong to (nearest mean), and how far is it from the mean? For data and generated samples both should agree: the weights 0.5 / 0.3 / 0.2 and the spread of each blob.

In [ ]:
def blob_stats(x):
    cos = ((toy.mu[None].float().transpose(-1, -2) @ x[:, None].float()).diagonal(dim1=-2, dim2=-1).sum(-1) - 1) / 2
    angle = torch.arccos(cos.clamp(-1, 1))
    k = angle.argmin(1)
    return k, angle.gather(1, k[:, None])[:, 0]


k_data, a_data = blob_stats(data[:3000])
k_gen, a_gen = blob_stats(generated[:, 0])
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for j, ax in enumerate(axes):
    ax.hist(np.rad2deg(a_data[k_data == j].numpy()), bins=40, density=True, alpha=0.5, label=f"data ({(k_data == j).float().mean():.2f})")
    ax.hist(np.rad2deg(a_gen[k_gen == j].numpy()), bins=40, density=True, alpha=0.5, label=f"generated ({(k_gen == j).float().mean():.2f})")
    ax.set_title(f"blob {j}: sigma = {float(toy.sigma[j])}, weight = {float(toy.w[j])}")
    ax.set_xlabel("angle to the mean (degrees)")
    ax.legend(title="share of samples", fontsize=8)
fig.tight_layout()
plt.show()

## 8. The likelihood: the probability flow ODE

The reverse SDE gives samples, but not densities. For every diffusion there is a *deterministic* flow with the same marginals $p_t$ (Song et al. 2021), the **probability flow ODE**. Since the noising has no drift, it is
$$\frac{dx}{dt} = -\tfrac12 \beta(t)\, \nabla \log p_t(x) =: V(x, t).$$
Along a trajectory of a flow the log-density changes by minus the divergence of the velocity (the continuity equation), $\frac{d}{dt}\log p_t(x_t) = -\operatorname{div} V(x_t, t)$. Integrating from the data ($t = \varepsilon$) to $t = 1$, where $p_1$ is uniform ($\log p_1 = 0$):
$$\log p_\varepsilon(x) = \underbrace{\log p_1(x_1)}_{= 0} + \int_\varepsilon^1 \operatorname{div} V(x_t, t)\, dt.$$

**The divergence on SO(3)^J.** Write $V = \sum_i \xi_i E_i$ with the frame fields $E_i(x) = x\,\hat e_i$ (one per joint and axis). These fields are *Killing* fields of the metric (they move rotations rigidly), so they are divergence free, and
$$\operatorname{div} V = \sum_i E_i(\xi_i) = \sum_i \frac{\partial}{\partial \delta_i}\, \xi_i\big(x \exp(\hat\delta)\big)\Big|_{\delta = 0}:$$
the trace of the Jacobian of the velocity coordinates with respect to a small rotation of the input. The code computes this Jacobian exactly with forward-mode automatic differentiation (`torch.func.jacfwd`, vectorized over the batch with `vmap`): $3J$ directional derivatives per point. The ODE is solved with the midpoint rule in the frame of the rotation.

In [ ]:
show_source(RSGM.velocity_and_divergence)
show_source(RSGM.log_prob)

**Checking the likelihood.** First with an *oracle*: an RSGM whose score is the exact score of the noised toy mixture (no network). Its ODE likelihood must equal the true density, which tests the method itself. Then the trained model, which is off by what the network has not learned.

In [ ]:
class Oracle(RSGM):
    # RSGM with the exact score of the noised toy mixture instead of the network.
    def score(self, x, t):
        return true_score(x, t, schedule=self).to(x.dtype)


oracle = Oracle(1, hidden=[4], beta_f=10.0, likelihood_steps=200).double()
test = toy.sample(500, torch.Generator().manual_seed(7)).reshape(-1, 1, 3, 3)
true_ll = toy.log_prob(test[:, 0], s=float(oracle.s(torch.tensor(oracle.eps))))
with torch.no_grad():
    oracle_ll = oracle.log_prob(test)
    model_ll = toy_model.log_prob(test.float()).double()

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(true_ll, oracle_ll, s=6, label=f"oracle (exact score): error {(oracle_ll - true_ll).mean():+.3f}")
ax.scatter(true_ll, model_ll, s=6, label=f"trained network: error {(model_ll - true_ll).mean():+.3f}")
lo, hi = float(true_ll.min()), float(true_ll.max())
ax.plot([lo, hi], [lo, hi], "k--", lw=1)
ax.set_xlabel("true log-density")
ax.set_ylabel("log-density from the probability flow ODE")
ax.legend(fontsize=8)
plt.show()
print(f"mean log-density of the test points: true {true_ll.mean():.3f}, oracle {oracle_ll.mean():.3f}, trained {model_ll.mean():.3f}")

**The number of ODE steps.** The integral is approximated with a finite number of midpoint steps. The learned score is stiff near $t = 0$ (it scales like $1/\sqrt{s}$), so too few steps bias the likelihood upwards. For real poses we measured: 100 steps ~0.3 nats too high, 200 ~0.06, 400 converged. On the toy:

In [ ]:
steps_list = [10, 25, 50, 100, 200, 400]
means = []
for steps in steps_list:
    toy_model.likelihood_steps = steps
    with torch.no_grad():
        means.append(toy_model.log_prob(test[:200].float()).mean().item())
toy_model.likelihood_steps = 200
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.semilogx(steps_list, means, "o-")
ax.set_xlabel("ODE steps")
ax.set_ylabel("mean log-likelihood")
ax.set_title("convergence of the likelihood in the number of steps")
plt.show()

## 9. The real thing: a pose prior trained on AMASS

The same model with $J = 19$ joints, trained with `train_rsgm.ipynb` on AMASS (100k steps of batch 512, ~17 minutes on an Apple GPU). Here we load the EMA weights of a trained run, generate poses and look at them as skeletons (forward kinematics of SMPL), next to real test poses. This section is skipped if there is no trained run.

In [ ]:
from hp.data.amass_cfg import AmassJoints4Learning
from hp.evaluation.data import load_reference
from hp.models.io import load_model
from hp.utils.skeleton_forward_kinematics import SkelForwardKinematics, perform_forward_kinematics

RUNS = ["~/data/experiments/rsgm/run_0001", "~/data/experiments/rsgm_c30/run_0001"]  # the first that exists is used
run = next((Path(r).expanduser() for r in RUNS if (Path(r).expanduser() / "weights_ema.pt").is_file()), None)
print(f"trained run: {run}")

In [ ]:
PARENTS = SkelForwardKinematics().parents[:22]


def draw_skeleton(ax, joints, color):
    # Joint positions [22, 3] of SMPL, seen from the front.
    for j in range(1, 22):
        p = PARENTS[j]
        ax.plot([joints[j, 0], joints[p, 0]], [joints[j, 1], joints[p, 1]], color=color, lw=2)
    ax.set_aspect("equal")
    ax.axis("off")


if run is not None:
    prior = load_model(run, device=DEVICE, weights="weights_ema")
    torch.manual_seed(0)
    poses = prior.sample(8)
    real = load_reference("test", "~/data/AMASS_rotations", "original")
    real = real[torch.randperm(len(real), generator=torch.Generator().manual_seed(0))[:8]]
    fig, axes = plt.subplots(2, 8, figsize=(16, 5))
    for row, (batch, color, label) in enumerate([(real, "C0", "real test poses"), (poses, "C1", "generated by RSGM")]):
        joints = perform_forward_kinematics(batch.float(), AmassJoints4Learning)
        for i in range(8):
            draw_skeleton(axes[row, i], joints[i].numpy(), color)
        axes[row, 0].set_title(label, loc="left")
    fig.tight_layout()
    plt.show()

    with torch.no_grad():
        ll_real = prior.log_prob(real.float())
        twisted = real.clone()
        twisted[:, [j.name for j in AmassJoints4Learning].index("LeftLeg")] = roma.rotvec_to_rotmat(torch.tensor([-1.5, 0.0, 0.0]))  # knee bent backwards
        ll_twisted = prior.log_prob(twisted.float())
    print("log-likelihood of the real poses:      ", np.round(ll_real.numpy(), 1))
    print("same poses, left knee bent backwards:   ", np.round(ll_twisted.numpy(), 1))

The prior gives an implausible pose (a knee bent the wrong way) a much lower log-likelihood. That, and its gradient, is what makes it useful in pose estimation: an optimizer can add $-\lambda \log p(x)$ to its objective, or follow the learned score at a small $t$, which approximates $\nabla \log p$.

For the evaluation against the HuProSO3 normalizing flow (Wasserstein distance, recall, precision, log-likelihood on the AMASS test split) see `rsgm_huproso_comparison.ipynb`.

## 10. Exercises

1. **Time convention.** Change the random walk of section 2 to steps $\sqrt{2\Delta s}\, z$. Which IGSO(3) time does it match now, and what is the generator of this process?
2. **The series.** How many terms of `log_density_series` are needed for an error below $10^{-6}$ at $s = 0.01$? Why does the image sum need only $|k| \le 4$?
3. **The weighting.** Train the toy model with the unweighted loss $\|s_\theta - \nabla\log p_{t|0}\|^2$ (remove the std factors in `RSGM.loss`). What happens to the training, and to the score at small $t$ (section 6)?
4. **The end of the noising.** Train with $\beta_f = 2$. Compare the oracle likelihood (section 8) with the truth: where does the error come from, and how does it depend on $\beta_f$?
5. **Sampling steps.** Generate with 10, 25 and 100 steps of the sampler. How do the blob shares and spreads (section 7) change?
6. **Deterministic sampling.** Generate by integrating the probability flow ODE from uniform rotations instead of the SDE (use `RSGM.velocity` and the midpoint rule, backwards in time). Compare with the SDE samples.
7. **Hutchinson's estimator.** Replace the exact divergence by $\mathbb{E}[\epsilon^\top J \epsilon]$ with one random $\epsilon$ (one Jacobian-vector product instead of $3J$). How noisy is the likelihood per pose, and how noisy is its mean over 500 poses?
8. **A pose prior in action.** Take a real pose, perturb some joints strongly and "denoise" it by gradient ascent on the learned score at $t = 0.05$ (section 9). Does it return to a plausible pose?

**Further reading.** De Bortoli et al., *Riemannian Score-Based Generative Modelling*, NeurIPS 2022. Song et al., *Score-Based Generative Modeling through Stochastic Differential Equations*, ICLR 2021. Leach et al., *Denoising Diffusion Probabilistic Models on SO(3) for Rotational Alignment*, 2022 (IGSO(3)). Dünkel et al., *Normalizing Flows on the Product Space of SO(3) Manifolds for Probabilistic Human Pose Modeling*, CVPR 2024 (HuProSO3).